<a href="https://colab.research.google.com/github/RevathyRamalingam/machineLearning/blob/main/Module8_DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd


In [ ]:
from tensorflow.keras.preprocessing.image import load_img

In [ ]:
import os

os.system("wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip")


0

In [ ]:
import zipfile

with zipfile.ZipFile('data.zip', 'r') as zip_ref:
    zip_ref.extractall()  # Specify the folder where you want to extract the files
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(torch.__version__)

2.9.0+cu126


In [ ]:
import os
from torch.utils.data import Dataset

class HairDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torchvision import transforms
input_size = 200

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.RandomRotation(10),           # Rotate up to 10 degrees
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0)),  # Zoom
    transforms.RandomHorizontalFlip(),       # Horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
from torch.utils.data import DataLoader

train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transforms
)

test_dataset = HairDataset(
    data_dir='./data/test',
    transform=test_transforms
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# Define the CNN model class (already defined in your code)
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 100 * 100, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the model and move it to the correct device
model = CNNModel()
model.to(device)

# Define the optimizer and loss function
optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
criterion = nn.BCELoss()


# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    running_loss = 0.0
    # Initialize test_total and test_correct for each epoch or before the loop if cumulative
    train_total = 0 # Initialize test_total
    train_correct = 0 # Initialize test_correct
    for i, (inputs_batch, labels_batch) in enumerate(train_loader):
        # Move batch to the correct device
        inputs_batch = inputs_batch.to(device)
        labels_batch = labels_batch.to(device)

        # Convert labels to float type
        labels_batch = labels_batch.float() # FIX: Convert labels to float

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs_batch)

        # Compute the loss
        loss = criterion(outputs.squeeze(), labels_batch.squeeze())

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()

        # Get predictions
        # Assuming binary classification, outputs are probabilities, convert to 0 or 1
        predicted = (outputs > 0.5).float() # Thresholding for binary predictions

        # Update total and correct predictions
        train_total += labels_batch.size(0)
        # Compare predicted with labels_batch after squeezing and ensure types match
        train_correct += (predicted.squeeze() == labels_batch.squeeze()).sum().item()

    # Calculate average training loss and accuracy for the epoch
    # These calculations were incorrectly placed inside the batch loop and using 'test_loss'/'test_acc'
    # They should be calculated per epoch for training data.
    train_loss = running_loss / len(train_loader)
    train_acc = train_correct / train_total

    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
    #print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

    # Optionally, save the model every epoch
    # torch.save(model.state_dict(), f"cnn_model_epoch_{epoch+1}.pth")

# After training, you can evaluate the model using validation data (if available)

# Test or inference
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradients for inference
    test_inputs = torch.randn(10, 3, 200, 200).to(device)  # Random test data (10 samples)
    outputs = model(test_inputs)
    predictions = (outputs > 0.5).float()  # Convert to binary predictions (0 or 1)

print("Predictions:", predictions)


Epoch 1/5
  Train Loss: 0.6684, Train Acc: 0.6042
Epoch 2/5
  Train Loss: 0.6416, Train Acc: 0.6841
Epoch 3/5
  Train Loss: 0.6826, Train Acc: 0.6292
Epoch 4/5
  Train Loss: 0.5655, Train Acc: 0.6941
Epoch 5/5
  Train Loss: 0.5358, Train Acc: 0.7079
Predictions: tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]], device='cuda:0')
